<a href="https://colab.research.google.com/github/xEzIxX/AI-Class/blob/master/week12/imdb_sentiment_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout

import re

In [9]:
# IMDB 영화 리뷰 데이터셋 불러오기
imdb = keras.datasets.imdb

# 자주 등장하는 상위 10,000개 단어만 사용
vocab_size = 10000

# 훈련 데이터와 테스트 데이터 다운로드
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

In [10]:
# 훈련 데이터와 테스트 데이터 개수 확인
print(len(x_train), len(x_test))

# 첫 번째 리뷰 확인
# 단어가 아니라 정수 인덱스로 저장되어 있음
print(x_train[0])

# 첫 번째 리뷰와 두 번째 리뷰의 길이 확인
print(len(x_train[0]), len(x_train[1]))

# 첫 번째, 두 번째 리뷰의 정답 라벨 확인
# 1은 긍정, 0은 부정
print(y_train[0], y_train[1])

25000 25000
[1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
218 189
1 0


In [11]:
# 긍정 리뷰와 부정 리뷰가 각각 몇 개인지 확인
np.unique(y_train, return_counts=True)

(array([0, 1]), array([12500, 12500]))

In [12]:
# 단어를 정수 인덱스로 바꿔주는 딕셔너리 가져오기
word_to_index = imdb.get_word_index()

# 특수 토큰을 넣기 위해 기존 인덱스를 3칸씩 뒤로 밀기
word_to_index = {k: (v + 3) for k, v in word_to_index.items()}

# 특수 토큰 추가
word_to_index["<PAD>"] = 0      # 길이를 맞출 때 사용하는 값
word_to_index["<START>"] = 1    # 문장 시작
word_to_index["<UNK>"] = 2      # 알 수 없는 단어
word_to_index["<UNUSED>"] = 3   # 사용하지 않는 값

# 정수 인덱스를 다시 단어로 바꾸기 위한 딕셔너리
index_to_word = dict([(value, key) for (key, value) in word_to_index.items()])

In [13]:
# 첫 번째 리뷰를 실제 단어 문장으로 복원해서 출력
print(" ".join([index_to_word[index] for index in x_train[0]]))

<START> this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert <UNK> is an amazing actor and now the same being director <UNK> father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for <UNK> and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also <UNK> to the two little boy's that played the <UNK> of norman and paul they were just brilliant children are often left out of the <UNK> list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for wha

In [16]:
# 리뷰마다 길이가 다르므로 길이를 100으로 통일
max_len = 100

# 길이가 100보다 길면 자르고, 짧으면 0으로 채움
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

# 길이가 100으로 맞춰졌는지 확인
print(len(x_train[0]), len(x_train[1]))

100 100


In [17]:
# 패딩 처리된 첫 번째 리뷰 확인
print(x_train[0])

[1415   33    6   22   12  215   28   77   52    5   14  407   16   82
    2    8    4  107  117 5952   15  256    4    2    7 3766    5  723
   36   71   43  530  476   26  400  317   46    7    4    2 1029   13
  104   88    4  381   15  297   98   32 2071   56   26  141    6  194
 7486   18    4  226   22   21  134  476   26  480    5  144   30 5535
   18   51   36   28  224   92   25  104    4  226   65   16   38 1334
   88   12   16  283    5   16 4472  113  103   32   15   16 5345   19
  178   32]


In [18]:
# 순차 신경망 모델 생성
model = Sequential()

# 단어 인덱스를 64차원 벡터로 변환
model.add(Embedding(vocab_size, 64, input_length=max_len))

# 2차원 데이터를 1차원으로 펼침
model.add(Flatten())

# 은닉층
model.add(Dense(64, activation="relu"))

# 과적합 방지를 위해 일부 뉴런을 랜덤하게 끔
model.add(Dropout(0.5))

# 출력층
# sigmoid는 0~1 사이 값 출력
model.add(Dense(1, activation="sigmoid"))

# 모델 구조 확인
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
# 모델 학습 방식 설정
model.compile(
    loss="binary_crossentropy",   # 긍정/부정 같은 이진 분류에 사용
    optimizer="adam",             # 가중치 업데이트 방법
    metrics=["accuracy"]          # 정확도 확인
)

# 모델 학습
history = model.fit(
    x_train,
    y_train,
    batch_size=64,
    epochs=20,
    verbose=1,
    validation_data=(x_test, y_test)
)

Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 26ms/step - accuracy: 0.7596 - loss: 0.4679 - val_accuracy: 0.8350 - val_loss: 0.3609
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - accuracy: 0.9354 - loss: 0.1776 - val_accuracy: 0.8339 - val_loss: 0.4059
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.9926 - loss: 0.0302 - val_accuracy: 0.8362 - val_loss: 0.5153
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.9990 - loss: 0.0065 - val_accuracy: 0.8380 - val_loss: 0.5967
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 8s 21ms/step - accuracy: 0.9998 - loss: 0.0020 - val_accuracy: 0.8398 - val_loss: 0.6421
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 20ms/step - accuracy: 0.9995 - loss: 0.0028 - val_accuracy: 0.8321 - val_loss: 0.6979
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 1.0000 - loss: 9.7941e-04 - val_accuracy: 0.8361 - val_loss: 0.7333
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - accuracy: 1.0000 - loss: 4.0735e-0

In [20]:
# 테스트 데이터로 모델 성능 평가
results = model.evaluate(x_test, y_test, verbose=2)

# [손실값, 정확도] 출력
print(results)

782/782 - 2s - 3ms/step - accuracy: 0.8235 - loss: 1.2114
[1.2113821506500244, 0.8234800100326538]


In [21]:
# 직접 작성한 영화 리뷰
review = """
What can I say about this movie that was already said?
It is my favorite time travel sci-fi, adventure epic comedy in the 80's
and I love this movie to death! When I saw this movie I was thrown out by its theme.
An excellent sci-fi, adventure epic, I LOVE the 80s.
It's simple the greatest time travel movie ever happened in the history of world cinema.
I love this movie to death, I love, LOVE, love it!
"""

In [22]:
# 알파벳, 숫자, 공백만 남기고 나머지 특수문자 제거
review = re.sub("[^0-9a-zA-Z ]", "", review).lower()

# 리뷰를 정수 인덱스로 바꿔 저장할 리스트
review_encoding = []

# 리뷰를 단어 단위로 나누어 정수 인덱스로 변환
for w in review.split():
    # 딕셔너리에 없는 단어는 <UNK>로 처리
    index = word_to_index.get(w, 2)

    # 상위 10,000개 단어 안에 있으면 그대로 사용
    if index < vocab_size:
        review_encoding.append(index)

    # 10,000개 밖의 단어는 <UNK>로 처리
    else:
        review_encoding.append(word_to_index["<UNK>"])

In [23]:
# 모델에 넣기 위해 길이를 100으로 맞춤
test_input = pad_sequences([review_encoding], maxlen=max_len)

# 긍정일 확률 예측
value = model.predict(test_input)[0][0]

# 0.5보다 크면 긍정, 아니면 부정으로 판단
if value > 0.5:
    print("긍정적인 리뷰입니다.")
else:
    print("부정적인 리뷰입니다.")

# 예측값 출력
print("예측값:", value)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step
긍정적인 리뷰입니다.
예측값: 1.0
